# 2 — Tester le pipeline V4 sur l'acquisition synthétique

Ce notebook reprend les cellules numériques de
`04_acdc_hardware_continuous_MRzero_V4.ipynb`. Les chemins et l'audit des
métadonnées sont adaptés : les headers de l'acquisition réelle 015 ne sont
pas utilisés pour décrire le fantôme. La vérité terrain n'entre pas dans
les reconstructions, l'estimation des cartes ou le choix des variantes.

Exécuter d'abord `01_generer_acquisition_MRzero.ipynb`.


In [ ]:
from pathlib import Path
import sys, json, hashlib
from datetime import datetime
import numpy as np
import matplotlib.pyplot as plt
import torch
from IPython.display import display, Markdown

PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
                    if (p/"Experiment").is_dir() and (p/"MRzero_validation").is_dir())
for folder in (PROJECT_ROOT/"Experiment", PROJECT_ROOT/"MRzero_validation"):
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))
from twin_inputs import read_twin_inputs, prepare_echo_object, frame_acquisition, build_candidates, model_arguments
from twin_geometry import compare_magnitude, fft_origin_reverse
from continuous_reconstruction import ContinuousReconstruction
from epi_twin import NativeEPITwin
from b0_regularization import regularize_b0_local_plane

WORK_DIR = PROJECT_ROOT/"Mrzero_simulation"
DATASET_NAME = "acquisition_synthetic_twix_001"
INPUT_DIR = WORK_DIR/DATASET_NAME
FRAME_ROTATION_DEG = 180  # Transformation commune des coordonnées ET du k-space.
OBJECT_THRESHOLD_FRACTION = 0.04
BACKGROUND_CORNER_M = 0.085
B0_SIGMA_PIXELS = 1.0  # Échelle fixée sur la grille native, sans ajustement au scan EPI.
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ADC_CHUNK = 128
OUTPUT_DIR = WORK_DIR/"pipeline_validation"/datetime.now().strftime("synthetic_%Y%m%d_%H%M%S_%f")
print("Entrée :", INPUT_DIR)
print("Simulation :", DEVICE, "| sorties :", OUTPUT_DIR)

def table(rows, headings):
    display(Markdown("| "+" | ".join(headings)+" |\n| "+" | ".join(["---"]*len(headings))+" |\n"+
                     "\n".join("| "+" | ".join(str(v) for v in row)+" |" for row in rows)))

def show(fig):
    plt.show()
    plt.close(fig)


## 1. Lire et vérifier les acquisitions

Le B0 est reconstruit depuis les produits complexes entre échos. La GRE et l'EPI
sont reconstruites avec les coordonnées ADC du `.seq`, sans rotation préalable des images.
Les phases des récepteurs déjà présentes dans le Twix ne sont pas appliquées une seconde fois.

In [ ]:
inputs = read_twin_inputs(INPUT_DIR)
acquisition = inputs["acquisition"]
raw = inputs["b0_raw"]
field = inputs["field"]
echo_times = inputs["echo_times_s"]
print("EPI :", inputs["paths"]["scan"].name)
print("Échos B0 [ms] :", echo_times*1000)
print("Matrices : GRE", inputs["gre"]["magnitude"].shape, "B0", field["b0Hz"].shape,
      "EPI", inputs["scan"].shape)
manifest = inputs["manifest"]
print("Données synthétiques :", manifest["format"])
print("Modèle RF :", manifest["model"]["RF"])
print("Vérité terrain utilisée par ce pipeline :", manifest["ground_truth_used_by_pipeline"])
assert not manifest["ground_truth_used_by_pipeline"]


## 2. Une rotation physique cohérente, indépendante de la résolution

Une rotation de tableau `rot90(..., 2)` autour de son centre ne conserve pas l'origine
FFT sur les matrices paires. Réattribuer ensuite les anciens X/Y crée un décalage
d'un pixel, différent à 32², 64² et 128².

Ici, le repère peut être retourné à 180° en appliquant **X→−X, Y→−Y et k→−k ensemble**.
Les valeurs B0 restent attachées aux mêmes spins. Le signal acquis reste identique.
La reconstruction mesurée utilise elle aussi ce k-space transformé. Cette opération
ne prouve ni ne modifie le placement physique des bobines ACDC.

In [ ]:
table([(n, 200/n, "origine FFT conservée par transformation commune") for n in (32,64,128)],
      ["Matrice", "Décalage de l'ancienne rotation fixe [mm]", "V4"])
frame_acq = frame_acquisition(acquisition, FRAME_ROTATION_DEG)
operator = ContinuousReconstruction(frame_acq, "cpu", torch.float64)
measured_coils = operator.numpy_image(np.moveaxis(inputs["scan_diagnostics"]["corrected_kspace"],1,0))
measured = np.sqrt(np.sum(abs(measured_coils)**2, axis=0))
expected_measured = fft_origin_reverse(inputs["scan"]) if FRAME_ROTATION_DEG == 180 else inputs["scan"]
frame_reconstruction_error = np.linalg.norm(measured-expected_measured)/np.linalg.norm(measured)
assert frame_reconstruction_error < 2e-4
X_epi, Y_epi = operator.X_m, operator.Y_m
print("Erreur relative du changement de repère :", frame_reconstruction_error)
fig, axes = plt.subplots(1,3,figsize=(13,4),layout="constrained")
for ax, image, title in zip(axes, [inputs["gre"]["magnitude"], field["b0Hz"], measured],
                           ["GRE native 32²", "B0 natif [Hz]", "EPI synthétique, repère commun"]):
    im=ax.imshow(image,origin="lower",cmap="RdBu_r" if "B0" in title else "gray")
    ax.set_title(title)
    fig.colorbar(im,ax=ax,shrink=.8)
show(fig)

## 3. Amplitude et décroissance à partir des trois échos natifs

Le modèle ajusté est `A(TE) = A_ref × exp[−R2* × (TE−TE_ref)]`, avec `TE_ref = TE₁`.
L'amplitude reste pondérée par la saturation GRE et la réception ; elle n'est pas une
mesure de densité de protons. Les trois échos ne permettent pas de déterminer T1.

Le bruit complexe est estimé uniquement dans les coins explicitement choisis et affichés.
La puissance RSS est corrigée de ce bruit. L'ajustement pondéré fournit une incertitude
propagée jusqu'au TE EPI. Une pente mal déterminée est conservée pour diagnostic,
mais n'est pas imposée à la simulation.

Le taux effectif contient le déphasage non résolu à 128². Les variations B0 entre
positions natives sont simulées séparément. La dispersion suivant z n'est pas mesurée.
Un ajustement statistiquement précis sur 3,64 ms ne démontre pas un comportement
mono-exponentiel jusqu'à 98 ms.

In [ ]:
fit = prepare_echo_object(inputs, OBJECT_THRESHOLD_FRACTION, BACKGROUND_CORNER_M)
report = fit["report"]
table([(key,report[key]) for key in ["object_voxels","simulated_voxels","excluded_missing_B0_voxels",
      "excluded_missing_B0_amplitude_fraction","fit_valid_voxels","target_reliable_voxels","fallback_voxels"]],
      ["Contrôle", "Valeur"])
print("Ajustement global conservateur :",json.dumps(fit["pooled"],indent=2))
fig, axes = plt.subplots(2,3,figsize=(14,8),layout="constrained")
rate_max = np.percentile(fit["R2star_sInv"][fit["object_support"]],95)
panels = [(fit["first_echo_amplitude"],"Amplitude au premier écho","gray",None),
          (fit["object_support"]+2*fit["noise_mask"],"Objet et coins utilisés pour le bruit","viridis",None),
          (fit["simulation_support"],"Objet avec B0 mesuré fiable","gray",None),
          (np.where(fit["object_support"],fit["R2star_sInv"],np.nan),"R2* effectif [s⁻¹]","magma",rate_max),
          (fit["target_log_std"],"Incertitude log-amplitude à TE EPI","magma",1),
          (fit["target_reliable_mask"],"Extrapolation statistiquement acceptable","gray",None)]
for ax,(image,title,cmap,vmax) in zip(axes.ravel(),panels):
    im=ax.imshow(image,origin="lower",cmap=cmap,vmin=0,vmax=vmax)
    ax.set_title(title)
    fig.colorbar(im,ax=ax,shrink=.8)
show(fig)

In [ ]:
pool=fit["pooled"]
times=np.linspace(echo_times[0],acquisition["TE_s"],250)
curve=pool["fitted_amplitude_at_reference"]*np.exp(-pool["R2star_sInv"]*(times-echo_times[0]))
fig,axes=plt.subplots(1,2,figsize=(12,4),layout="constrained")
for ax in axes:
    ax.plot(times*1000,curve,"-",label="Extrapolation effective globale")
    ax.plot(echo_times*1000,pool["mean_RSS_echoes"],"o",label="Trois échos synthétiques")
    ax.set(xlabel="TE [ms]",ylabel="Amplitude RSS moyenne")
    ax.legend()
axes[0].set_xlim(0,7)
axes[0].set_title("Intervalle réellement mesuré")
axes[1].axvline(acquisition["TE_s"]*1000,color="red",linestyle="--",label="TE EPI")
axes[1].set_title("Distance jusqu'au TE EPI")
show(fig)

### Régulariser le bruit du B0 sans déplacer le champ

La phase accumulée à 98 ms amplifie le bruit d'une carte B0 estimée avec des
échos rapprochés. Nous comparons aussi une version régularisée à une échelle
fixée à un pixel natif (1,56 mm), sans ajuster cette échelle à l'EPI.
Un plan local pondéré conserve les gradients linéaires, y compris près du masque.
La moyenne du champ est conservée. Aucune valeur manquante n'est remplie.
Cette hypothèse de régularité spatiale peut atténuer des variations réelles très
locales : le B0 brut et les résultats sans régularisation restent disponibles.


In [ ]:
spacing_mm = [abs(raw['Y_m'][1,0]-raw['Y_m'][0,0])*1000,
              abs(raw['X_m'][0,1]-raw['X_m'][0,0])*1000]
b0_regularized = regularize_b0_local_plane(field['b0Hz'],field['mask'],
    sigma_pixels=B0_SIGMA_PIXELS,spacing_mm=spacing_mm)
fit['regularized_B0_Hz'] = b0_regularized['b0Hz']
fit['regularized_B0_change_Hz'] = b0_regularized['change_Hz']
report['B0_regularization'] = b0_regularized['report']
print(json.dumps(b0_regularized['report'],indent=2))
fig,axes=plt.subplots(1,3,figsize=(13,4),layout='constrained')
for ax,image,title in zip(axes,[field['b0Hz'],b0_regularized['b0Hz'],b0_regularized['change_Hz']],
                         ['B0 brut [Hz]','B0 régularisé [Hz]','Changement [Hz]']):
    im=ax.imshow(image,origin='lower',cmap='RdBu_r')
    ax.set_title(title)
    fig.colorbar(im,ax=ax,shrink=.8)
show(fig)

## 4. Comparaisons contrôlées, toutes simulées par MRzero

Le cas `legacy_GRE64` est un diagnostic de l'ancienne convention : GRE 32² interpolée
à 64² et rotation des tableaux sur une grille fixe. Il ne reproduit pas tous les
réglages de ROI d'un ancien résultat sauvegardé.

Les autres cas changent successivement la géométrie, l'amplitude native et la relaxation.
Le domaine de simulation ne dépend plus de la couverture des bobines ACDC. Les spins
sans B0 fiable sont explicitement exclus, et leur fraction d'amplitude est indiquée.
Le B0 manquant n'est pas remplacé par un champ homogène dans l'objet.

L'amplitude est ancrée au **TE du premier écho GRE**, puis évolue jusqu'aux ADC EPI.
La convention ancienne d'amplitude ancrée au TE EPI n'est pas réutilisée pour les fits.
La coupe est une amplitude intégrée sur 5 mm : en l'absence de B0 mesuré suivant z,
8 positions identiques suivant l'épaisseur donneraient exactement le même signal
sans ACDC et à kz=0. Aucun gradient B0 suivant z n'est inventé.
Les amplitudes reçoivent une fois le poids d'aire du pixel objet par rapport au
pixel EPI : 1 pour la grille 64², 1/4 pour la grille 128².

In [ ]:
candidates = build_candidates(inputs,fit,FRAME_ROTATION_DEG)
table([(c["name"],c["amplitude"].shape,int(c["support_mask"].sum()),c["note"]) for c in candidates],
      ["Variante","Grille objet","Spins actifs","Hypothèse"])
results={}
for candidate in candidates:
    name=candidate["name"]
    print("MRzero :",name,flush=True)
    twin=NativeEPITwin(**model_arguments(candidate),device=DEVICE,sample_chunk=ADC_CHUNK)
    ideal=twin.simulate(b0_enabled=False)
    measured_twin=twin.simulate(b0_enabled=True)
    if candidate["legacy_output_reindex"]:
        # Revenir du diagnostic ancien au repère commun du benchmark.
        for output in (ideal,measured_twin):
            image=np.rot90(output["image"],2)
            output["image"]=fft_origin_reverse(image) if FRAME_ROTATION_DEG==180 else image
    results[name]={"ideal":ideal,"measured":measured_twin}
    del twin
    if DEVICE=="cuda": torch.cuda.empty_cache()

## 5. Mesurer la ressemblance sans déformer les images

La mesure principale utilise tout le FOV 64², identique pour tous les cas. Un seul
gain positif est ajusté par variante pour absorber les différences d'unités entre
acquisitions ; aucun déplacement ni déformation n'est ajusté. Les contours et les
gradients sont évalués séparément de la NRMSE d'intensité.

Ces scores décrivent la ressemblance à l'acquisition utilisée pour cette comparaison.
Choisir une variante sur ces scores est une sélection sur cette acquisition,
pas une validation indépendante de sa capacité de généralisation.

In [ ]:
metrics={}
for name,result in results.items():
    metrics[name]=compare_magnitude(measured,result["measured"]["image"],X_m=X_epi,Y_m=Y_epi)
table([(name,round(m["NRMSE_percent"],3),round(m["edge_gradient_NRMSE_percent"],3),
        round(m["contour_Dice"],4),round(m["centroid_distance_mm"],3),round(m["gain"],6))
       for name,m in metrics.items()],
      ["Variante","NRMSE [%]","Erreur gradients [%]","Dice contour","Écart centroïde [mm]","Gain global"])
modern_names=[name for name in results if name!="legacy_GRE64"]
best_name=min(modern_names,key=lambda name:metrics[name]["NRMSE_percent"])
print("Meilleure NRMSE parmi les variantes modernes sur ce scan :",best_name)

In [ ]:
scale=measured.max()
fig,axes=plt.subplots(len(results),4,figsize=(15,3.6*len(results)),layout="constrained")
for row,(name,result) in enumerate(results.items()):
    gain=metrics[name]["gain"]
    images=[gain*result["ideal"]["image"]/scale,gain*result["measured"]["image"]/scale,
            measured/scale,abs(gain*result["measured"]["image"]-measured)/scale]
    titles=["Twin B0 nul","Twin B0 mesuré","EPI synthétique sans ACDC","Différence absolue"]
    for col,(image,title) in enumerate(zip(images,titles)):
        ax=axes[row,col]
        ax.imshow(image,origin="lower",cmap="magma" if col==3 else "gray",vmin=0,vmax=.5 if col==3 else 1.2)
        ax.set_title(title)
        ax.set_xticks([]);ax.set_yticks([])
    axes[row,0].set_ylabel(name+"\nNRMSE="+str(round(metrics[name]["NRMSE_percent"],2))+" %")
show(fig)

In [ ]:
best=results[best_name]
gain=metrics[best_name]["gain"]
image=gain*best["measured"]["image"]
dx_mm=abs(X_epi[0,1]-X_epi[0,0])*1000
dy_mm=abs(Y_epi[1,0]-Y_epi[0,0])*1000
extent=[X_epi.min()*1000-dx_mm/2,X_epi.max()*1000+dx_mm/2,Y_epi.min()*1000-dy_mm/2,Y_epi.max()*1000+dy_mm/2]
fig,axes=plt.subplots(1,3,figsize=(15,4),layout="constrained")
axes[0].imshow(measured,origin="lower",extent=extent,cmap="gray")
axes[0].contour(X_epi*1000,Y_epi*1000,image,levels=[.2*measured.max()],colors="orange",linewidths=1)
axes[0].set_title("Orange : contour du meilleur twin")
for column in [24,32,40]:
    axes[1].plot(Y_epi[:,column]*1000,measured[:,column]/scale,label=f"Synthétique x={X_epi[0,column]*1000:.0f} mm")
    axes[1].plot(Y_epi[:,column]*1000,image[:,column]/scale,"--",label="Twin")
axes[1].set(xlabel="Y phase [mm]",ylabel="Amplitude normalisée",title="Profils dans la direction de phase")
axes[1].legend(fontsize=7)
axes[2].imshow((image-measured)/scale,origin="lower",extent=extent,cmap="RdBu_r",vmin=-.5,vmax=.5)
axes[2].set_title("Résidu signé / maximum synthétique")
show(fig)

## 6. Contrôler le changement de repère sur un vrai signal MRzero

Le signal d'un objet et celui du même objet avec X/Y et k tous inversés doivent
être identiques. Seule la représentation de l'image change. Cette vérification
ne déduit pas le signe du B0 ou le placement des bobines d'une minimisation d'erreur.

In [ ]:
candidate=next(c for c in candidates if c["name"]=="echo128_no_decay")
args=model_arguments(candidate)
args["acquisition"]=acquisition
args["X_m"]=raw["X_m"]
args["Y_m"]=raw["Y_m"]
native_model=NativeEPITwin(**args,device=DEVICE,sample_chunk=ADC_CHUNK)
native_result=native_model.simulate(b0_enabled=True)
frame_result=results["echo128_no_decay"]["measured"]
frame_signal_error=np.linalg.norm(native_result["kdata"]-frame_result["kdata"])/np.linalg.norm(native_result["kdata"])
frame_image_expected=fft_origin_reverse(native_result["image"]) if FRAME_ROTATION_DEG==180 else native_result["image"]
frame_image_error=np.linalg.norm(frame_result["image"]-frame_image_expected)/np.linalg.norm(frame_result["image"])
assert frame_signal_error < 5e-5
assert frame_image_error < 2e-4
print("Signal MRzero : erreur de changement de repère",frame_signal_error)
print("Image MRzero : erreur de changement de repère",frame_image_error)
del native_model
if DEVICE=="cuda":torch.cuda.empty_cache()

## 7. Sauvegarder les résultats et l'objet caractérisé

Les résultats contiennent les deux jumeaux, les données complexes simulées, les
cartes d'amplitude et de taux, leurs masques de fiabilité, les scores, les sources
et les paramètres. L'objet reste une amplitude effective GRE ; les différences
de saturation T1 entre GRE multi-échos et EPI ne sont pas identifiées par trois échos.

Une cartographie volumique B0, une mesure de relaxation à des TE plus longs et une
calibration de la réponse des gradients permettraient de lever les limites restantes.
Aucun courant n'est optimisé ou exporté pendant cette validation du modèle de base.

In [ ]:
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
arrays={"measured_EPI":measured,"measured_coil_images":measured_coils,
        "X_epi_m":X_epi,"Y_epi_m":Y_epi,"native_X_m":raw["X_m"],"native_Y_m":raw["Y_m"],
        "native_B0_Hz":field["b0Hz"],"echo_times_s":echo_times,
        "native_complex_echoes":raw["coil_images"]}
for key,value in fit.items():
    if isinstance(value,np.ndarray): arrays["echo_fit_"+key]=value
for name,result in results.items():
    for kind in ("ideal","measured"):
        arrays[name+"_"+kind+"_image"]=result[kind]["image"]
        arrays[name+"_"+kind+"_kdata"]=result[kind]["kdata"]
np.savez_compressed(OUTPUT_DIR/"twin_validation.npz",**arrays)
summary={"dataset":str(INPUT_DIR),"baseline_scan":inputs["paths"]["scan"].name,
    "source_sha256":inputs["source_sha256"],"frame_rotation_deg":FRAME_ROTATION_DEG,
    "frame_operation":"joint spin coordinates and k-space transform; no fixed-grid data rotation",
    "frame_signal_relative_error":float(frame_signal_error),"frame_image_relative_error":float(frame_image_error),
    "best_on_calibration_scan":best_name,"independent_validation":False,
    "metrics":metrics,"echo_fit":report,
    "simulation_reports":{name:{kind:result[kind]["report"] for kind in ("ideal","measured")}
                          for name,result in results.items()},
    "ACDC_optimisation_run":False,"hardware_export":False,
    "remaining_assumptions":["Effective GRE amplitude includes saturation and reception; not PD.",
        "Three short echoes do not identify T1 or establish long-TE monoexponential validity.",
        "B0 is measured in one 5-mm slice; no measured through-slice distribution.",
        "Nominal Pulseq trajectory and ideal RF excitation; no measured gradient response."]}
def serialisable(value):
    if isinstance(value,dict):return {str(k):serialisable(v) for k,v in value.items()}
    if isinstance(value,(list,tuple)):return [serialisable(v) for v in value]
    if isinstance(value,np.ndarray):return serialisable(value.tolist())
    if isinstance(value,np.generic):return serialisable(value.item())
    if isinstance(value,float) and not np.isfinite(value):return None
    return value
(OUTPUT_DIR/"twin_validation.json").write_text(json.dumps(serialisable(summary),indent=2,ensure_ascii=False,allow_nan=False),encoding="utf-8")
print("Résultats :",OUTPUT_DIR)

## Interpréter ce test

Les différences entre le meilleur jumeau et l'EPI synthétique mesurent ici
les erreurs de reconstruction, de couverture, de fit et d'extrapolation du
pipeline. Même avec un simulateur exact, des échos courts n'identifient pas
parfaitement la décroissance jusqu'à 98 ms. La régularisation B0 change aussi
légèrement le champ connu.

Le notebook de génération a vérifié séparément le moteur EPI avec les cartes
exactes. Ce test de cohérence numérique ne démontre pas la fidélité du modèle
aux acquisitions réelles, au profil RF ou aux imperfections des gradients.
